# 08 — Local MLflow evidence and promotion decisions

        **Estimated time:** 40 minutes<br>
        **Prerequisites:** 07 — Frozen regression evaluation<br>
        **Learner-produced evidence:** a local run record and an adopt/reject/inconclusive assessment

        ## Learning objectives

        - Inspect local experiment lineage without a tracking server.
- Require comparable complete reports before making a promotion decision.
- Apply absolute output gates in addition to relative macro-F1 improvement.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Local tracking, not cloud tracking

MLflow uses a repository-local SQLite database and local artifact root.
Runs can record dataset fingerprints, model revision, adapter/config
hashes, predictions, reports, and measured metrics without credentials.


In [ ]:
import mlflow

from aai_local_finetuning.evaluation import (
    BaselineEvaluation,
    PromotionThresholds,
    decide_lora_promotion,
)
from aai_local_finetuning.learning import load_report, load_support_splits
from aai_local_finetuning.settings import PROJECT_ROOT, load_settings
from aai_local_finetuning.tracking import configure_local_mlflow

settings = load_settings()
configure_local_mlflow(settings)
runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=20)
runs[
    [
        column
        for column in (
            "run_id",
            "tags.mlflow.runName",
            "tags.run_purpose",
            "metrics.intent/macro_f1",
            "metrics.output/json_schema_validity_rate",
        )
        if column in runs.columns
    ]
]

## Log a learner checkpoint

This small run records that the notebook reached the decision stage. It
does not masquerade as a frozen evaluation run and does not overwrite
official evidence.


In [ ]:
with mlflow.start_run(run_name="notebook-decision-checkpoint") as run:
    mlflow.set_tags(
        {
            "run_purpose": "learner_checkpoint",
            "execution_mode": "offline_local",
        }
    )
    mlflow.log_metric("notebook_stage_complete", 1.0)
    checkpoint_run_id = run.info.run_id
checkpoint_run_id

## Inventory full notebook reports

Promotion requires all meaningful baselines and the LoRA change on the
complete frozen set with identical fingerprints. Partial reports, missing
methods, or mismatched fingerprints force `inconclusive`.


In [ ]:
splits = load_support_splits(settings)
report_dir = PROJECT_ROOT / "artifacts" / "notebook" / "evaluation"
required_methods = (
    "majority",
    "keyword-rule",
    "basic",
    "strong",
    "few_shot",
    "lora-change",
)
report_paths = {
    method: report_dir / f"full-{method}-report.json" for method in required_methods
}
report_status = {method: path.is_file() for method, path in report_paths.items()}
report_status

## Apply the decision contract

Defaults require schema validity ≥ 0.98, response-policy compliance ≥
0.95, unsupported-intent rate = 0, and macro F1 strictly above the
strongest meaningful baseline. Majority is retained as a floor but is
excluded from the meaningful-baseline competition.


In [ ]:
thresholds = PromotionThresholds()
if all(report_status.values()):
    loaded_reports = {
        method: load_report(path) for method, path in report_paths.items()
    }
    fingerprints = {report.evaluation_fingerprint for report in loaded_reports.values()}
    counts = {report.total_examples for report in loaded_reports.values()}
    complete_and_comparable = len(fingerprints) == 1 and counts == {len(splits.test)}
else:
    loaded_reports = {}
    complete_and_comparable = False

if complete_and_comparable:
    assessment = decide_lora_promotion(
        change_name="bitext-structured-output-lora-v1",
        change_report=loaded_reports["lora-change"],
        baselines=[
            BaselineEvaluation(
                name=name,
                report=loaded_reports[name],
                meaningful=name != "majority",
            )
            for name in required_methods
            if name != "lora-change"
        ],
        thresholds=thresholds,
    ).model_dump(mode="json")
else:
    assessment = {
        "decision": "inconclusive",
        "reasons": [
            "all six methods must be scored on the complete frozen set",
            "report counts and evaluation fingerprints must match",
        ],
        "available_reports": report_status,
    }
assessment

## Exercise — defend the decision

Write a short rationale that cites the strongest meaningful baseline,
macro-F1 gain, schema gate, policy gate, and unsupported-intent gate—or
names the missing evidence that makes the decision inconclusive.


In [ ]:
decision_rationale = (
    "The current notebook run remains inconclusive unless all six full "
    "reports share the frozen fingerprint; partial metrics are not promotion evidence."
)
assert any(
    word in decision_rationale.lower() for word in ("adopt", "reject", "inconclusive")
)
decision_rationale

**Hint:** a decision is not a summary adjective. It is a reproducible
function over baseline, change, result, thresholds, and comparability.


## Checkpoint

You have completed the Bitext lifecycle without treating training as
success or using a partial run as promotion evidence.

**Next:** `09_capstone_policy_dataset.ipynb` asks a different question:
when should deterministic policy, not a language model, own the truth?
